<a href="https://colab.research.google.com/github/hajonghyun/installPytorch_study/blob/main/11_custom_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset,DataLoader,random_split
from torchvision import datasets, transforms

In [ ]:
# data 다운 <AI 허브>
# https://aihub.or.kr/aihubdata/data/view.do?currMenu=115&topMenu=100&aihubDataSe=realm&dataSetSn=126

!pip install gdown
!gdown https://drive.google.com/uc?id=14lAjaR2dRp5p5kEsm5GnwNM9KH-VgoOq -O 대화체.xlsx

Downloading...
From: https://drive.google.com/uc?id=14lAjaR2dRp5p5kEsm5GnwNM9KH-VgoOq
To: /content/대화체.xlsx
100% 9.57M/9.57M [00:00<00:00, 25.5MB/s]


# 🚀 Why CustomDataset? (굳이 클래스로 감싸는 이유)

**"작은 데이터는 그냥 써도 되지만, 현업의 대용량 데이터는 '규격(Standard)'이 없으면 터진다."**

단순히 엑셀 파일을 `for`문으로 돌리는 것과 PyTorch의 `Dataset` 클래스를 사용하는 것의 결정적인 차이는 **메모리(Memory)**, **효율(Efficiency)**, **확장성(Scalability)**에 있다.

---

## 1. 핵심 이유 3가지

### ① 메모리 효율성: Lazy Loading (지연 로딩)
- **Raw Data 방식:** 데이터를 변수에 한 번에 다 로드함.
    - *문제점:* 데이터가 100GB라면? RAM(16GB)이 바로 터짐 (**OOM: Out Of Memory**).
- **CustomDataset 방식:** 리스트에 **'파일 경로(주소)'**만 들고 있음.
    - *해결:* `__getitem__`이 호출되는 순간에만 **딱 1개** 데이터를 디스크에서 읽어옴.
    - 1TB 데이터도 16GB 램에서 학습 가능.

### ② `DataLoader`와의 환상적인 콤비 (Batch & Shuffle)
- 딥러닝은 데이터를 하나씩 학습하지 않고 **배치(Batch)** 단위로 학습함.
- **Raw Data 방식:** 배치 묶기, 순서 섞기(Shuffle), 텐서 변환 등을 매번 직접 구현해야 함 (코드 복잡도 ↑).
- **CustomDataset 방식:**
    - `Dataset`은 "하나를 꺼내는 법(`__getitem__`)"만 정의함.
    - 나머지 **배치 만들기, 셔플, 멀티 프로세싱(병렬 처리)**은 `DataLoader`가 알아서 수행함.

### ③ 전처리(Transform)의 일관성 및 모듈화
- 학습 시 이미지 크기 조절, 정규화(Normalization), 증강(Augmentation)이 필수임.
- **CustomDataset 방식:** 데이터를 꺼낼 때(`__getitem__`) 전처리를 적용해서 내보냄.
    - 원본 데이터를 훼손하지 않고, 학습 시점에만 변형을 가함.
    - 데이터셋 교체 시, 모델 학습 코드를 건드리지 않고 `Dataset` 클래스만 갈아끼우면 됨.

---

## 2. 직관적 비유: 뷔페(Buffet) vs 코스 요리

| 구분 | Raw Data (통으로 로드) | CustomDataset (Lazy Loading) |
| :--- | :--- | :--- |
| **비유** | 뷔페 음식 **100접시를 한 번에** 내 테이블(RAM)에 가져오기 | 주문할 때마다 주방에서 **한 접시씩** 새로 만들어 오기 |
| **결과** | 테이블 공간 부족으로 **식사 불가 (Memory Overflow)** | 테이블이 작아도 **무한히 식사 가능** |
| **특징** | 데이터가 작을 때만 가능 (Toy Project) | **현업/실무 표준 (Production)** |

---

## 3. 코드 비교 (Before & After)

### ❌ Bad Practice (Raw Data Loop)
데이터가 크면 여기서 메모리 에러 발생.

```python
# 데이터를 메모리에 통째로 로딩
all_images = load_all_images("data_folder/")

# 직접 배치 만들고 셔플해야 함 (복잡)
for i in range(0, len(all_images), batch_size):
    batch = all_images[i : i+batch_size]
    # ... 학습 코드 ...
```

### ✅ Good Practice (PyTorch Standard)
데이터가 아무리 커도 메모리 안전.

```python
class MyDataset(Dataset):
    def __getitem__(self, idx):
        # 필요할 때 1개만 로드 (Lazy Loading)
        return load_one_image(self.paths[idx])

# DataLoader가 배치, 셔플, 병렬 처리를 자동 수행
loader = DataLoader(MyDataset(paths), batch_size=32, shuffle=True)

for batch in loader:
    # ... 깔끔한 학습 코드 ...
```

---

## 만능 코드 청사진 (Blueprint)

어떤 데이터를 만나든 이 틀에 채워 넣으면 됨.

```python
from torch.utils.data import Dataset

class CustomDataset(Dataset):
    def __init__(self, data_path, transform=None):
        """ 1. 초기화: 데이터 경로 로드 """
        self.data = data_path
        self.transform = transform

    def __len__(self):
        """ 2. 길이 반환: 총 데이터 개수 """
        return len(self.data)

    def __getitem__(self, idx):
        """ 3. 데이터 추출: idx번째 데이터 1개 반환 """
        # (1) Raw Data 가져오기
        sample = self.data[idx]
        
        # (2) 전처리 (이미지 변환 등)
        if self.transform:
            sample = self.transform(sample)
            
        # (3) (입력, 정답) 튜플 반환
        return sample
```

---
---
---

In [ ]:
class CustomDataset(Dataset):
    def __init__(self,data):
        self.data = data

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        return self.data.loc[idx,'원문'], self.data.loc[idx,'번역문']

In [ ]:
BATCH_SIZE = 8

data = pd.read_excel('대화체.xlsx')
custom_DS = CustomDataset(data)
train_DS, val_DS, test_DS, _ = random_split(custom_DS, [32,16,8,len(custom_DS)-32-16-8])

train_DL = DataLoader(train_DS, batch_size=BATCH_SIZE, shuffle=True)
val_DL = DataLoader(val_DS, batch_size=BATCH_SIZE, shuffle=True)
test_DL = DataLoader(test_DS, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
# train_DL 테스트
src_texts, trg_texts = next(iter(train_DL))

print(src_texts)
print(trg_texts)
print(len(src_texts))
print(len(trg_texts))

('그렇다면 중국에서 최대한 몇 년까지 일할 수 있나요?', '큰일이네, 그럼 해결될 때까지 한참 기다려야 하잖아.', '그 점에 전적으로 동의합니다.', '역사 시간에 그 소리를 듣고 분노했던 기억이 나네요.', '저는 분명히 번호를 한 개만 있는데 제 번호 밑에 모르는 번호가 또 있어요.', '우선 고기 판매대에서 간 소고기를 좀 사야 해.', '남성용 양말이 500박스, 어린이용 양말이 400박스, 스타킹이 400박스입니다.', '아이와 함께 저녁 식사를 하려고 하는데 매콤 찜닭은 아주 매울까요?')
('Then how many years can you work in China?', 'Oh my god, we will have to wait a while until it’s resolved.', 'I completely agree on that point.', 'I remember hearing it in history class and I got really angry.', "I have only one number, but there's another number I don't know about under my number.", 'First, I need to grab some ground beef from the meat store.', "500 boxes of men's socks, 400 boxes of children's socks, 400 boxes of stockings.", "I'm going to have dinner with my child, will the spicy steamed chicken be a bit too hot?")
8
8


In [ ]:
# val_DL 테스트
src_texts, trg_texts = next(iter(val_DL))

print(src_texts)
print(trg_texts)
print(len(src_texts))
print(len(trg_texts))

('저희 500mL 잔에 얼음물 하나랑 얼음만 따로도 주세요.', '포장해드리고 발송을 시작하겠습니다.', '땀을 너무 많이 흘려서 가끔 빈혈이 오기도 하는 것 같아서요.', '우리 업체는 판매 전문이라 소프트웨어 설치는 하지 않습니다.', '문 앞 키오스크에서 번호표를 뽑아서 기다려 주세요.', '안녕하세요, 하자 상품 반품 관련해서 전화드렸어요.', '그럼 그 부분은 알겠다고 전달 드리고 3월에 확인해 봅시다.', '아니요, 혹시 차를 더 작은 사이즈로 변경할 수 있을까요?')
('We would like a 500ml beer as well as some iced water and some ice separately.', 'We will deliver it after gift wrapping.', 'I think I sometimes get anemic because I sweat too much.', 'We are a company specializing in sales, so we do not install software.', 'Please take a number from the kiosk in front of the door and wait.', "Hello, I'm calling about the return of the defective product.", "Alright, tell them that’s fine and let's check again back in March.", 'No, could I change the car to a smaller size?')
8
8


In [ ]:
# test_DL 테스트
src_texts, trg_texts = next(iter(test_DL))

print(src_texts)
print(trg_texts)
print(len(src_texts))
print(len(trg_texts))

('주말에서 평일로 변경하시는 경우에는 가능하세요.', '가끔 청소해야 하는 경우도 있는데 괜찮을까요?', '그래요? 도움이 필요하면 말씀해주세요. 도울 수 있을 만큼 도와드릴게요.', '여기 "휘낭시에"라고 쓰여 있는 디저트는 어떤 디저트인가요?', '그러면 주황색으로 바꿔주세요.', '나도 어제 너처럼 집에 일찍 갔어야 했는데.', '아직 두시밖에 안 됐어, 왜 이렇게 피곤한지 모르겠네.', '그래야겠어요, 그리고 앞으로 답변을 빨리 달라고 이야기해봐야겠어요.')
('It is possible in case you are changing it from the weekend to a weekday.', 'Sometimes you have to clean and would that be okay?', 'Really? Please let me know if you need any help. I can give you a hand.', 'What kind of dessert is the dessert that\'s written as, "Financier"?', 'Then please change it to the orange colored one.', 'I should have gone home early yesterday like you.', "It is only 2 o'clock, I don't know why I am so tired.", "I should, and I'll have to tell him to give me a quick answer from now.")
8
8


# 이미지 데이터일 때.

In [ ]:
class CustomDataset_image(Dataset):
    def __init__(self, X, Y, transform=None):
        self.X = X
        self.Y = Y
        self.transform = transform

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self,idx):
        x = self.X[idx]
        y = self.Y[idx]
        if self.transform is not None:
            x = self.transform(x)
        return x,y


In [ ]:
class SubsetWithTransform(Dataset): # random_split 으로 나눈 다음 transform 따로 주고 싶을 때
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
        self.classes = subset.dataset.classes

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        x, y = self.subset[idx]
        if self.transform:
            x = self.transform(x)
        return x, y

In [ ]:
transform = transforms.ToTensor()
train_DS = CustomDataset_image(np.random.randn(10000,32,32,3), np.random.randint(1, 4, size=10000), transform=transform)
train_DL = DataLoader(train_DS, batch_size=2, shuffle=True)

print(type(train_DS.X)) # Dataset 안에서는 여전히 ndarray
x_batch, y_batch = next(iter(train_DL))
print(x_batch.shape) # 개채행열로 순서가 바뀐 모습
print(y_batch.shape)
print(type(x_batch))
# 참고 사항: ToTensor()를 하지 않아도 (transform=transform을 지우고 확인) tensor로 바뀌어 있음.
# 왜냐하면, __getitem__이 반환한 개별 샘플(예: x, y)이 DataLoader에 의해 모아지는데,
# 이때 내장된 collate 함수가 ndarray를 tensor로 변환하기 때문에 batch로 "모아진" 데이터는 tensor임
# 즉, __getitem__으로 인덱싱 해오고 collate_fn을 통해 묶는다!

<class 'numpy.ndarray'>
torch.Size([2, 3, 32, 32])
torch.Size([2])
<class 'torch.Tensor'>
